# 01 - Data Preprocessing
## Loan Risk Assessment System

**Objective:** Load the raw loan prediction dataset, inspect its structure, clean missing values, and persist a cleaned dataset for downstream use.

**Workflow:**
1. Load dataset
2. Inspect shape, schema, missing values, statistics
3. Drop identifier column
4. Compare Credit_History imputation strategies
5. Clean dataset (mode/median imputation)
6. Save cleaned dataset

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src import config
from src.data_loader import load_dataset, inspect_dataset
from src.preprocessing import clean_dataset, compare_credit_history_strategies

pd.set_option('display.max_columns', None)

## Step 1: Load Dataset

In [2]:
raw_df = load_dataset(config.RAW_DATA_PATH)
raw_df.head()

[2026-07-31 19:26:15] INFO - src.data_loader - Dataset loaded successfully from /home/claude/AIML-BonusProject-Loan-Risk-Assessment/Dataset/loan_prediction.csv with shape (614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP000001,Male,Yes,3+,Graduate,No,11135,1773.0,700.0,360.0,1.0,Urban,Y
1,LP000002,Male,No,0,Not Graduate,No,8800,0.0,700.0,360.0,1.0,Rural,Y
2,LP000003,Female,No,0,Graduate,No,19532,0.0,NaN,360.0,1.0,Rural,N
3,LP000004,Male,Yes,1,Graduate,No,2406,5935.0,700.0,360.0,1.0,Semiurban,Y
4,LP000005,Male,Yes,0,Graduate,Yes,11522,3071.0,700.0,360.0,1.0,Rural,Y


## Step 2: Dataset Inspection
Shape, schema info, missing values, and descriptive statistics.

In [3]:
inspect_dataset(raw_df)


DATASET SHAPE
Rows: 614, Columns: 13

DATASET INFO
<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             602 non-null    str    
 2   Married            611 non-null    str    
 3   Dependents         599 non-null    str    
 4   Education          614 non-null    str    
 5   Self_Employed      584 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         593 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     565 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 62.5 KB

MISSING VALUES
                  missing_count  missing_pct
Credit_History

## Step 3: Credit History Imputation Strategy Comparison

We compare **two distinct strategies** for handling missing `Credit_History` values:
- **Strategy 1 (Mode):** Fill missing values with the most frequent credit history value.
- **Strategy 2 (Unknown category):** Encode missing values as a distinct sentinel category (-1), treating 'missingness' itself as informative.

The comparison below evaluates approval rates under each strategy to decide which is carried forward into production.

In [4]:
strategy_comparison = compare_credit_history_strategies(raw_df)
strategy_comparison


CREDIT HISTORY IMPUTATION STRATEGY COMPARISON
Missing values in 'Credit_History': 49 (7.98%)
[2026-07-31 19:26:15] INFO - src.preprocessing - [Strategy 1: Mode] Imputed 'Credit_History' missing values with mode: 1.0


[2026-07-31 19:26:15] INFO - src.preprocessing - [Strategy 2: Unknown Category] Imputed 'Credit_History' missing values with sentinel value: -1


                Strategy_1_Mode_ApprovalRate  \
Credit_History                                 
-1.0                                     NaN   
 0.0                                0.232558   
 1.0                                0.829545   

                Strategy_2_UnknownCategory_ApprovalRate  
Credit_History                                           
-1.0                                           0.836735  
 0.0                                           0.232558  
 1.0                                           0.828810  

Conclusion: The 'Unknown category' strategy preserves the signal that missing credit history is itself predictive, and is the strategy carried forward into the production pipeline, since applicants with -1 (unknown) credit history exhibit an approval rate distinct from both 0 and 1 groups.


,Strategy_1_Mode_ApprovalRate,Strategy_2_UnknownCategory_ApprovalRate
Credit_History,,
-1.0,NaN,0.836735
0.0,0.232558,0.232558
1.0,0.829545,0.828810


**Conclusion:** The 'Unknown category' strategy is selected for the production pipeline because applicants with unknown credit history show a distinctly different approval rate from both the 'good' (1.0) and 'bad' (0.0) groups — this signal would be lost under mode imputation.

## Step 4: Run Full Cleaning Pipeline

This drops `Loan_ID`, applies mode imputation to categorical fields, median imputation to numeric fields, and the 'Unknown category' strategy to `Credit_History`.

In [5]:
cleaned_df = clean_dataset(raw_df)
print(f'Cleaned dataset shape: {cleaned_df.shape}')
print(f'Remaining missing values: {cleaned_df.isnull().sum().sum()}')
cleaned_df.head()

[2026-07-31 19:26:15] INFO - src.preprocessing - Starting data cleaning pipeline...


[2026-07-31 19:26:15] INFO - src.preprocessing - Dropped identifier column: 'Loan_ID'


[2026-07-31 19:26:15] INFO - src.preprocessing - Imputed missing values in 'Gender' with mode: Male


[2026-07-31 19:26:15] INFO - src.preprocessing - Imputed missing values in 'Married' with mode: Yes


[2026-07-31 19:26:15] INFO - src.preprocessing - Imputed missing values in 'Dependents' with mode: 0


...

[2026-07-31 19:26:15] INFO - src.preprocessing - Imputed missing values in 'Loan_Amount_Term' with median: 360.0


[2026-07-31 19:26:15] INFO - src.preprocessing - [Strategy 2: Unknown Category] Imputed 'Credit_History' missing values with sentinel value: -1


[2026-07-31 19:26:15] INFO - src.preprocessing - Data cleaning complete. Final shape: (614, 12)


Cleaned dataset shape: (614, 12)
Remaining missing values: 0


,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,Yes,3+,Graduate,No,11135,1773.0,700.0,360.0,1.0,Urban,Y
1,Male,No,0,Not Graduate,No,8800,0.0,700.0,360.0,1.0,Rural,Y
2,Female,No,0,Graduate,No,19532,0.0,700.0,360.0,1.0,Rural,N
3,Male,Yes,1,Graduate,No,2406,5935.0,700.0,360.0,1.0,Semiurban,Y
4,Male,Yes,0,Graduate,Yes,11522,3071.0,700.0,360.0,1.0,Rural,Y


## Step 5: Save Cleaned Dataset

In [6]:
cleaned_df.to_csv(config.CLEANED_DATA_PATH, index=False)
print(f'Saved cleaned dataset to: {config.CLEANED_DATA_PATH}')

Saved cleaned dataset to: /home/claude/AIML-BonusProject-Loan-Risk-Assessment/outputs/cleaned_dataset.csv


## Conclusion

The raw dataset has been fully inspected and cleaned:
- The non-predictive `Loan_ID` column was dropped.
- Categorical missing values (`Gender`, `Married`, `Dependents`, `Self_Employed`) were imputed with their mode.
- Numeric missing values (`LoanAmount`, `Loan_Amount_Term`) were imputed with their median.
- `Credit_History` missing values were encoded as an informative 'Unknown' category (-1) after comparing it against simple mode imputation.

The cleaned dataset contains **zero missing values** and is ready for feature engineering in the next notebook (`02_EDA.ipynb`).